# DLBCL external validation — revision analyses (addendum A1–A3)
Inputs: committed output of `dlbcl_validation_clean` (parquets/CSVs) + `lenz-signatures` dataset.
Rebuilds frozen models from parquets (deterministic, seeded), then runs:
**A1** CV-tuned genes-only as sole de novo model (metrics recomputed) · **A2** reverse-direction de novo (train HMRN → validate GSE10846/GSE87371) · **A3** multiple imputation of IPI components.
Addendum deposited on OSF before execution: [link].

In [1]:
!pip -q install scikit-survival lifelines


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 35.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 80.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 8.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.


In [4]:
import glob
print(glob.glob('/kaggle/input/**/*', recursive=True))

['/kaggle/input/notebooks', '/kaggle/input/notebooks/lawin2593', '/kaggle/input/notebooks/lawin2593/dlbcl-notebook2', '/kaggle/input/notebooks/lawin2593/dlbcl-notebook2/GPL14951.soft.txt', '/kaggle/input/notebooks/lawin2593/dlbcl-notebook2/GSE31312_family.soft.gz', '/kaggle/input/notebooks/lawin2593/dlbcl-notebook2/GPL14951_family.soft.gz', '/kaggle/input/notebooks/lawin2593/dlbcl-notebook2/__results__.html', '/kaggle/input/notebooks/lawin2593/dlbcl-notebook2/gse181063_matrix.txt.gz', '/kaggle/input/notebooks/lawin2593/dlbcl-notebook2/gse181063_clin.csv', '/kaggle/input/notebooks/lawin2593/dlbcl-notebook2/gse181063_expr_genes.parquet', '/kaggle/input/notebooks/lawin2593/dlbcl-notebook2/__notebook__.ipynb', '/kaggle/input/notebooks/lawin2593/dlbcl-notebook2/GSE117556_family.soft.gz', '/kaggle/input/notebooks/lawin2593/dlbcl-notebook2/gse181063_expr_probes.parquet', '/kaggle/input/notebooks/lawin2593/dlbcl-notebook2/__output__.json', '/kaggle/input/notebooks/lawin2593/dlbcl-notebook2/gse

In [6]:
import pandas as pd, numpy as np, glob, os, sys, gzip, io, urllib.request
from sksurv.linear_model import CoxnetSurvivalAnalysis, CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored, cumulative_dynamic_auc
from sksurv.util import Surv
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

def find(name):
    h = glob.glob(f'/kaggle/input/**/{name}', recursive=True); return h[0] if h else None

expr_tr  = pd.read_parquet(find("gse10846_expr.parquet"))
clin_tr  = pd.read_csv(find("gse10846_clin.csv"), index_col=0)
expr_val = pd.read_parquet(find("gse181063_expr_genes.parquet"))
cf = find("gse181063_clin_fixed.csv")
if cf:
    clin_val = pd.read_csv(cf, index_col=0)
else:
    # fallback: re-parse from the series matrix in DLBCL_notebook2 output (per-cell key parsing)
    mpath = find("gse181063_matrix.txt.gz")
    if mpath is None:
        os.system("wget -q -O /kaggle/working/gse181063_matrix.txt.gz https://ftp.ncbi.nlm.nih.gov/geo/series/GSE181nnn/GSE181063/matrix/GSE181063_series_matrix.txt.gz")
        mpath = "/kaggle/working/gse181063_matrix.txt.gz"
    gsm_ids, char_rows = None, []
    with gzip.open(mpath, "rt", errors="ignore") as f:
        for line in f:
            if line.startswith("!Sample_geo_accession"):
                gsm_ids = [x.strip().strip('"') for x in line.rstrip("\n").split("\t")[1:]]
            elif line.startswith("!Sample_characteristics_ch1"):
                char_rows.append([x.strip().strip('"') for x in line.rstrip("\n").split("\t")[1:]])
            elif line.startswith("!series_matrix_table_begin"): break
    rec = {g:{} for g in gsm_ids}
    for rowv in char_rows:
        for gsm, cell in zip(gsm_ids, rowv):
            if ":" in cell:
                k,v = cell.split(":",1); k,v = k.strip().lower(), v.strip()
                if v: rec[gsm][k]=v
    clin_val = pd.DataFrame(rec).T
    clin_val.to_csv("/kaggle/working/gse181063_clin_fixed.csv")
    print("clin_val re-parsed from matrix")
print(expr_tr.shape, clin_tr.shape, expr_val.shape, clin_val.shape)

sys.path.append(os.path.dirname(find("lenz_signatures.py")))
from lenz_signatures import GCB_SYMBOLS, STROMAL1_SYMBOLS, STROMAL2_SYMBOLS, LENZ_ALIASES
print(len(GCB_SYMBOLS), len(STROMAL1_SYMBOLS), len(STROMAL2_SYMBOLS))

clin_val re-parsed from matrix
(22880, 420) (420, 10) (20817, 1310) (1310, 41)
36 260 61


## GSE87371 — load parquets (fallback: re-download)

In [7]:
p87 = find("gse87371_expr_genes.parquet")
if p87:
    expr3g = pd.read_parquet(p87)
    df3 = pd.read_csv(find("gse87371_clin.csv"), index_col=0)
    clin3 = pd.read_csv(find("gse87371_clin_raw.csv"), index_col=0)
    print("GSE87371 loaded from parquet:", expr3g.shape)
else:
    print("parquet missing -> re-downloading GSE87371 (~5 min)")
    m3path = "/kaggle/working/gse87371_matrix.txt.gz"
    os.system(f"wget -q -O {m3path} https://ftp.ncbi.nlm.nih.gov/geo/series/GSE87nnn/GSE87371/matrix/GSE87371_series_matrix.txt.gz")
    gsm_ids3, char_rows3 = None, []
    with gzip.open(m3path, "rt", errors="ignore") as f:
        for line in f:
            if line.startswith("!Sample_geo_accession"):
                gsm_ids3 = [x.strip().strip('"') for x in line.rstrip("\n").split("\t")[1:]]
            elif line.startswith("!Sample_characteristics_ch1"):
                char_rows3.append([x.strip().strip('"') for x in line.rstrip("\n").split("\t")[1:]])
            elif line.startswith("!series_matrix_table_begin"): break
    rec = {g:{} for g in gsm_ids3}
    for rowv in char_rows3:
        for gsm, cell in zip(gsm_ids3, rowv):
            if ":" in cell:
                k,v = cell.split(":",1); k,v = k.strip().lower(), v.strip()
                if v: rec[gsm][k]=v
    clin3 = pd.DataFrame(rec).T
    expr3 = pd.read_csv(m3path, sep="\t", comment="!", index_col=0, compression="gzip"); expr3.columns=[c.strip('"') for c in expr3.columns]
    resp = urllib.request.urlopen("https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPLnnn/GPL570/soft/GPL570_family.soft.gz")
    gz = gzip.GzipFile(fileobj=resp); buf=[]; it=False
    for raw in gz:
        line = raw.decode("utf-8", errors="ignore")
        if line.startswith("!platform_table_begin"): it=True; continue
        if line.startswith("!platform_table_end"): break
        if it: buf.append(line)
    resp.close()
    ann3 = pd.read_csv(io.StringIO("".join(buf)), sep="\t", low_memory=False)
    p2g = ann3.set_index("ID")["Gene Symbol"].astype(str).str.split(" /// ").str[0]
    expr3g = expr3.copy(); expr3g["gene"] = expr3g.index.map(p2g)
    expr3g = expr3g[expr3g["gene"].notna() & ~expr3g["gene"].isin(["nan",""])]
    v = expr3g.drop(columns="gene").var(axis=1); expr3g = expr3g.loc[v.groupby(expr3g["gene"]).idxmax().values].set_index("gene")
    df3 = pd.DataFrame(index=clin3.index)
    df3["time_yrs"] = pd.to_numeric(clin3["os_time"], errors="coerce")/12.0
    df3["event"] = 1 - pd.to_numeric(clin3["cens_os"], errors="coerce")   # verified: 1=censored
    df3["ipi_given"] = pd.to_numeric(clin3["ipi"], errors="coerce"); df3["coo"] = clin3["coo"].astype(str).str.upper()
    expr3g.to_parquet("/kaggle/working/gse87371_expr_genes.parquet"); df3.to_csv("/kaggle/working/gse87371_clin.csv"); clin3.to_csv("/kaggle/working/gse87371_clin_raw.csv")
keep3 = df3["time_yrs"].notna() & df3["event"].notna() & df3["ipi_given"].notna() & ~df3["coo"].isin(["PMBL"])
ids3 = df3.index[keep3].intersection(expr3g.columns)
print("GSE87371 eval n:", len(ids3), "| events:", int(df3.loc[ids3,"event"].sum()))

parquet missing -> re-downloading GSE87371 (~5 min)
GSE87371 eval n: 201 | events: 48


## Rebuild HMRN analysis frame, primary population, eval set

In [8]:
cv = clin_val.copy()
dfv = pd.DataFrame(index=cv.index)
dfv["time_yrs"] = pd.to_numeric(cv["os_followup_y"], errors="coerce")
dfv["event"]    = pd.to_numeric(cv["os_status"], errors="coerce")
dfv["age"]      = pd.to_numeric(cv["age_at_diagnosis"], errors="coerce")
dfv["ecog"]     = pd.to_numeric(cv["performance_status_ecog"], errors="coerce")
dfv["stage"]    = cv["stage"].astype(str).str.extract(r'(IV|III|II|I)')[0].map({"I":1,"II":2,"III":3,"IV":4})
dfv["ldh_raised"] = cv["ldh"].astype(str).str.lower().eq("raised").astype(float)
dfv.loc[~cv["ldh"].astype(str).str.lower().isin(["raised","normal"]), "ldh_raised"] = np.nan
dfv["extranodal"] = pd.to_numeric(cv["num_extranodal"], errors="coerce")
dfv["ipi_given"]  = pd.to_numeric(cv["ipi_score"], errors="coerce")
dfv["coo"]        = cv["pred_combine"]
dfv["qc_fail"]    = pd.to_numeric(cv["qc_fail"], errors="coerce")
dfv["regimen"]    = cv["firstline_regimen"]
dfv["curative"]   = pd.to_numeric(cv["curative_intent"], errors="coerce")
dfv["dlbcl"]      = cv["diagnostic_group"].astype(str).str.contains("DLBCL", na=False)
primary = (dfv["dlbcl"] & (dfv["qc_fail"]==0) & (dfv["curative"]==1)
           & dfv["regimen"].isin(["CHOP-R","CHOP-R/Bortezomib"]) & dfv["time_yrs"].notna() & dfv["event"].notna())
val_ids = dfv.index[primary].intersection(expr_val.columns)
yv = dfv.loc[val_ids]; eval_ids = val_ids[yv["ipi_given"].notna()]; ev = yv.loc[eval_ids]
print("Primary:", primary.sum(), "| eval (IPI available):", len(eval_ids), "| events:", int(ev["event"].sum()))

Primary: 730 | eval (IPI available): 552 | events: 221


## Rebuild frozen models on GSE10846 R-CHOP (deterministic)

In [9]:
common_genes = expr_tr.index.intersection(expr_val.index)
tr = clin_tr[(clin_tr["rchop"]==1) & clin_tr["time_yrs"].notna()].copy()
tr_ids = tr.index.intersection(expr_tr.columns); tr = tr.loc[tr_ids]
y_tr = Surv.from_arrays(tr["event"].astype(bool), tr["time_yrs"])
Xg = expr_tr.loc[common_genes, tr_ids].T; top = Xg.var().nlargest(2000).index
X_tr = StandardScaler().fit_transform(Xg[top])

# CV-tuned genes-only (A1: sole de novo model)
pm = CoxnetSurvivalAnalysis(l1_ratio=0.9, alpha_min_ratio=0.01, n_alphas=50, max_iter=100000).fit(X_tr, y_tr)
alphas = pm.alphas_; cvs = np.full((5,len(alphas)), np.nan)
for k,(tri,tei) in enumerate(KFold(5, shuffle=True, random_state=42).split(X_tr)):
    m = CoxnetSurvivalAnalysis(l1_ratio=0.9, alphas=alphas, max_iter=100000).fit(X_tr[tri], y_tr[tri])
    for j,a in enumerate(alphas):
        try: cvs[k,j] = concordance_index_censored(y_tr[tei]["event"], y_tr[tei]["time"], m.predict(X_tr[tei], alpha=a))[0]
        except Exception: pass
mc = np.nanmean(cvs, axis=0); best_alpha = alphas[np.nanargmax(mc)]
mA_cv = CoxnetSurvivalAnalysis(l1_ratio=0.9, alphas=[best_alpha], max_iter=100000).fit(X_tr, y_tr)
print(f"CV-tuned: alpha={best_alpha:.4f} internal C={np.nanmax(mc):.3f} nonzero={(mA_cv.coef_.ravel()!=0).sum()}")

# clinical Cox
tr_clin = tr[["age","ecog","stage","extranodal"]].copy(); tr_clin["ldh_bin"] = (tr["ldh_ratio"]>1).astype(float)
tr_clin_cc = tr_clin.dropna()
y_cc = Surv.from_arrays(tr.loc[tr_clin_cc.index,"event"].astype(bool), tr.loc[tr_clin_cc.index,"time_yrs"])
m_clin = CoxPHSurvivalAnalysis().fit(tr_clin_cc.values, y_cc)

# COO+IPI
tc = tr["coo"].astype(str).str.upper()
ci_tr = pd.DataFrame({"abc": tc.str.contains("ABC").astype(float), "unc": (~tc.str.contains("ABC|GCB")).astype(float), "ipi": tr["ipi"]}).dropna()
m3 = CoxPHSurvivalAnalysis().fit(ci_tr.values, Surv.from_arrays(tr.loc[ci_tr.index,"event"].astype(bool), tr.loc[ci_tr.index,"time_yrs"]))

# Lenz model + Lenz+IPI
def resolve(genes, index):
    out=[]
    for g in genes:
        alt = LENZ_ALIASES.get(g, g)
        for c in (g, alt):
            if c is not None and c in index: out.append(c); break
    return out
def lenz_score(expr_df, ids, standardize=False):
    vals=[]
    for gl in (GCB_SYMBOLS, STROMAL1_SYMBOLS, STROMAL2_SYMBOLS):
        g = resolve(gl, expr_df.index); vals.append(expr_df.loc[g, ids].mean(axis=0))
    gcb,s1,s2 = vals
    if standardize: gcb,s1,s2 = [(x-x.mean())/x.std() for x in (gcb,s1,s2)]
    return 8.11 - 0.419*gcb - 1.015*s1 + 0.675*s2
sc_tr = lenz_score(expr_tr, tr_ids); sc_tr_z = (sc_tr-sc_tr.mean())/sc_tr.std()
Xl = pd.DataFrame({"lenz": sc_tr_z, "ipi": tr["ipi"]}).dropna()
m_lenz_ipi = CoxPHSurvivalAnalysis().fit(Xl.values, Surv.from_arrays(tr.loc[Xl.index,"event"].astype(bool), tr.loc[Xl.index,"time_yrs"]))
print("Lenz training C:", round(concordance_index_censored(tr["event"].astype(bool), tr["time_yrs"], sc_tr.values)[0],3))
print("models rebuilt")

CV-tuned: alpha=0.0905 internal C=0.683 nonzero=27
Lenz training C: 0.692
models rebuilt


## Shared evaluation helpers

In [10]:
def boot_generic(name, e, t, s, ipi, n_boot=1000, seed=42):
    e,t,s,ipi = map(np.asarray, (e,t,s,ipi)); rng=np.random.default_rng(seed); ds=[]
    for _ in range(n_boot):
        i=rng.integers(0,len(e),len(e))
        try: ds.append(concordance_index_censored(e[i],t[i],s[i])[0]-concordance_index_censored(e[i],t[i],ipi[i])[0])
        except Exception: pass
    lo,hi=np.percentile(ds,[2.5,97.5])
    print(f"{name:42s} n={len(e)} C={concordance_index_censored(e,t,s)[0]:.3f} (IPI {concordance_index_censored(e,t,ipi)[0]:.3f}) dC={np.mean(ds):+.3f} [{lo:+.3f}, {hi:+.3f}]")
def row(name, ids, s):  boot_generic(name, dfv.loc[ids,"event"].astype(bool), dfv.loc[ids,"time_yrs"], s, dfv.loc[ids,"ipi_given"])
def boot3(name, ids, s): boot_generic(name, df3.loc[ids,"event"].astype(bool), df3.loc[ids,"time_yrs"], s, df3.loc[ids,"ipi_given"])
def ext_metrics(name, ids, s):
    ys = Surv.from_arrays(dfv.loc[ids,"event"].astype(bool), dfv.loc[ids,"time_yrs"])
    auc,_ = cumulative_dynamic_auc(ys, ys, np.asarray(s), [2.0,5.0])
    slope = CoxPHSurvivalAnalysis().fit(np.asarray(s).reshape(-1,1), ys).coef_[0]
    print(f"{name:34s} AUC2y={auc[0]:.3f} AUC5y={auc[1]:.3f} slope={slope:.2f}")

# external inputs on eval set
pred_cv_hmrn = pd.Series(mA_cv.predict(StandardScaler().fit_transform(expr_val.loc[top, eval_ids].reindex(top).T)), index=eval_ids)
pred_cv_87   = mA_cv.predict(StandardScaler().fit_transform(expr3g.loc[top, ids3].T))
vc = ev["coo"].astype(str).str.upper()
ci_v = pd.DataFrame({"abc": vc.eq("ABC").astype(float), "unc": (~vc.isin(["ABC","GCB"])).astype(float), "ipi": ev["ipi_given"]}).dropna()
ids4 = ci_v.index; pred3 = m3.predict(ci_v.values)
val_clin = ev[["age","ecog","stage","extranodal","ldh_raised"]]; val_ok = val_clin.notna().all(axis=1); ids_v = eval_ids[val_ok.values]
lp_v = m_clin.predict(val_clin.loc[ids_v].values.astype(float))
sc_h = lenz_score(expr_val, eval_ids, standardize=True); sc_h_z = (sc_h-sc_h.mean())/sc_h.std()
lenz_ipi_h = m_lenz_ipi.predict(np.c_[sc_h_z, ev["ipi_given"]])

## A1 — CV-tuned genes-only as sole de novo model

In [11]:
print("=== A1 ===")
row("Genes-only [CV-tuned] GSE181063", eval_ids, pred_cv_hmrn.values)
ext_metrics("Genes-only [CV-tuned]", eval_ids, pred_cv_hmrn.values)
ext_metrics("IPI", eval_ids, ev["ipi_given"].values)
ext_metrics("COO+IPI", ids4, pred3)
ext_metrics("Clinical Cox", ids_v, lp_v)
ext_metrics("Lenz+IPI", eval_ids, lenz_ipi_h)
boot3("Genes-only [CV-tuned] GSE87371", ids3, pred_cv_87)

=== A1 ===
Genes-only [CV-tuned] GSE181063            n=552 C=0.631 (IPI 0.685) dC=-0.056 [-0.103, -0.009]
Genes-only [CV-tuned]              AUC2y=0.671 AUC5y=0.673 slope=1.00
IPI                                AUC2y=0.740 AUC5y=0.721 slope=0.51
COO+IPI                            AUC2y=0.761 AUC5y=0.763 slope=0.92
Clinical Cox                       AUC2y=0.774 AUC5y=0.759 slope=0.90
Lenz+IPI                           AUC2y=0.795 AUC5y=0.759 slope=0.84
Genes-only [CV-tuned] GSE87371             n=201 C=0.662 (IPI 0.787) dC=-0.126 [-0.226, -0.033]


## A2 — Reverse direction: train genes-only on HMRN (n≈730), validate on GSE10846 R-CHOP and GSE87371
Pre-stated prediction (addendum): ΔC vs IPI ≤ 0 in both.

In [12]:
rev_ids = val_ids
y_rev = Surv.from_arrays(dfv.loc[rev_ids,"event"].astype(bool), dfv.loc[rev_ids,"time_yrs"])
print("Reverse training:", len(rev_ids), "| events:", int(dfv.loc[rev_ids,"event"].sum()))
Xh = expr_val.loc[common_genes, rev_ids].T; top_rev = Xh.var().nlargest(2000).index
X_rev = StandardScaler().fit_transform(Xh[top_rev])
pm_r = CoxnetSurvivalAnalysis(l1_ratio=0.9, alpha_min_ratio=0.01, n_alphas=50, max_iter=100000).fit(X_rev, y_rev)
ar = pm_r.alphas_; cvr = np.full((5,len(ar)), np.nan)
for k,(tri,tei) in enumerate(KFold(5, shuffle=True, random_state=42).split(X_rev)):
    m = CoxnetSurvivalAnalysis(l1_ratio=0.9, alphas=ar, max_iter=100000).fit(X_rev[tri], y_rev[tri])
    for j,a in enumerate(ar):
        try: cvr[k,j] = concordance_index_censored(y_rev[tei]["event"], y_rev[tei]["time"], m.predict(X_rev[tei], alpha=a))[0]
        except Exception: pass
mcr = np.nanmean(cvr, axis=0); a_best = ar[np.nanargmax(mcr)]
m_rev = CoxnetSurvivalAnalysis(l1_ratio=0.9, alphas=[a_best], max_iter=100000).fit(X_rev, y_rev)
print(f"best alpha={a_best:.4f} | internal CV C={np.nanmax(mcr):.3f} | nonzero genes={(m_rev.coef_.ravel()!=0).sum()}")

g10 = tr.index[tr["ipi"].notna()].intersection(expr_tr.columns)
p10 = m_rev.predict(StandardScaler().fit_transform(expr_tr.loc[top_rev, g10].T))
boot_generic("Reverse genes-only -> GSE10846 R-CHOP", tr.loc[g10,"event"].astype(bool), tr.loc[g10,"time_yrs"], p10, tr.loc[g10,"ipi"])
tr87 = top_rev.intersection(expr3g.index); print("genes present in GSE87371:", len(tr87), "/ 2000")
p87 = m_rev.predict(StandardScaler().fit_transform(expr3g.reindex(top_rev).loc[:, ids3].fillna(0).T))
boot3("Reverse genes-only -> GSE87371", ids3, p87)
ys10 = Surv.from_arrays(tr.loc[g10,"event"].astype(bool), tr.loc[g10,"time_yrs"])
print("calibration slope on GSE10846:", round(CoxPHSurvivalAnalysis().fit(p10.reshape(-1,1), ys10).coef_[0],2))

Reverse training: 730 | events: 318
best alpha=0.0451 | internal CV C=0.676 | nonzero genes=82
Reverse genes-only -> GSE10846 R-CHOP      n=164 C=0.708 (IPI 0.699) dC=+0.008 [-0.087, +0.101]
genes present in GSE87371: 2000 / 2000
Reverse genes-only -> GSE87371             n=201 C=0.713 (IPI 0.787) dC=-0.075 [-0.176, +0.021]
calibration slope on GSE10846: 1.11


## A3 — Multiple imputation of IPI components (all primary-population patients)

In [13]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
pop = dfv.loc[val_ids].copy()
coo_d = pd.get_dummies(pop["coo"].astype(str).str.upper(), prefix="coo").astype(float)
aux = pd.DataFrame({"event": pop["event"], "logt": np.log(pop["time_yrs"].clip(lower=0.01))}, index=pop.index)
comp = ["age","ecog","stage","ldh_raised","extranodal"]
M = pd.concat([pop[comp], coo_d, aux], axis=1)
print("missing per component:", pop[comp].isna().sum().to_dict())
genes_pop = mA_cv.predict(StandardScaler().fit_transform(expr_val.loc[top, pop.index].reindex(top).T))
sc_pop = lenz_score(expr_val, pop.index, standardize=True); sc_pop_z = (sc_pop-sc_pop.mean())/sc_pop.std()
vcp = pop["coo"].astype(str).str.upper(); abc_p = vcp.eq("ABC").astype(float).values; unc_p = (~vcp.isin(["ABC","GCB"])).astype(float).values
e, t = pop["event"].astype(bool).values, pop["time_yrs"].values

def eval_imp(seed):
    Mi = pd.DataFrame(IterativeImputer(random_state=seed, max_iter=20, sample_posterior=True).fit_transform(M), index=M.index, columns=M.columns)
    age=Mi["age"].values; ecog=Mi["ecog"].round().clip(0,4).values; stage=Mi["stage"].round().clip(1,4).values
    ldh=Mi["ldh_raised"].round().clip(0,1).values; ext=Mi["extranodal"].round().clip(lower=0).values
    ipi = (age>60).astype(int)+ldh+(ecog>=2).astype(int)+(stage>=3).astype(int)+(ext>=2).astype(int)
    out = {"COO+IPI": m3.predict(np.c_[abc_p,unc_p,ipi]), "Lenz+IPI": m_lenz_ipi.predict(np.c_[sc_pop_z.values,ipi]),
           "Clinical Cox": m_clin.predict(np.c_[age,ecog,stage,ext,ldh]), "Genes CV-tuned": genes_pop}
    res = {"IPI": (concordance_index_censored(e,t,ipi)[0], 0.0, 0.0)}
    for k,s in out.items():
        rng=np.random.default_rng(seed); ds=[]
        for _ in range(500):
            i=rng.integers(0,len(e),len(e)); ds.append(concordance_index_censored(e[i],t[i],s[i])[0]-concordance_index_censored(e[i],t[i],ipi[i])[0])
        res[k]=(concordance_index_censored(e,t,s)[0], float(np.mean(ds)), float(np.var(ds)))
    return res
runs=[eval_imp(s) for s in range(10)]
print(f"=== A3: MI, n={len(pop)}, events={int(pop['event'].sum())}, m=10 ===")
for k in ["IPI","COO+IPI","Lenz+IPI","Clinical Cox","Genes CV-tuned"]:
    Cs=np.array([r[k][0] for r in runs]); ds=np.array([r[k][1] for r in runs]); wv=np.array([r[k][2] for r in runs])
    se = 0 if k=="IPI" else np.sqrt(wv.mean() + 1.1*ds.var(ddof=1))
    print(f"{k:16s} C={Cs.mean():.3f}  dC={ds.mean():+.3f} [{ds.mean()-1.96*se:+.3f}, {ds.mean()+1.96*se:+.3f}]")

missing per component: {'age': 0, 'ecog': 59, 'stage': 82, 'ldh_raised': 141, 'extranodal': 96}
=== A3: MI, n=730, events=318, m=10 ===
IPI              C=0.678  dC=+0.000 [+0.000, +0.000]
COO+IPI          C=0.704  dC=+0.026 [+0.005, +0.048]
Lenz+IPI         C=0.701  dC=+0.023 [-0.003, +0.049]
Clinical Cox     C=0.706  dC=+0.028 [+0.012, +0.044]
Genes CV-tuned   C=0.628  dC=-0.050 [-0.094, -0.006]


## Persist inputs for future notebooks

In [14]:
expr3g.to_parquet("/kaggle/working/gse87371_expr_genes.parquet"); df3.to_csv("/kaggle/working/gse87371_clin.csv"); clin3.to_csv("/kaggle/working/gse87371_clin_raw.csv")
clin_val.to_csv("/kaggle/working/gse181063_clin_fixed.csv")
expr_tr.to_parquet("/kaggle/working/gse10846_expr.parquet"); clin_tr.to_csv("/kaggle/working/gse10846_clin.csv")
expr_val.to_parquet("/kaggle/working/gse181063_expr_genes.parquet")
print("all inputs persisted — commit this notebook and future notebooks need only its output")

all inputs persisted — commit this notebook and future notebooks need only its output
